In [33]:
# -----------------------
# Setup Dash for Jupyter
# -----------------------
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px
import base64

# Import my CRUD module
from animal_shelter import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

# -----------------------
# Connect to MongoDB
# -----------------------
username = "aacuser"
password = "Password123"

shelter = AnimalShelter(username, password)

animals = shelter.read({})
df = pd.DataFrame.from_records(animals)

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# -----------------------
# Load Logo
# -----------------------
image_filename = "/home/codio/workspace/code_files/Grazioso Salvare Logo.png"

with open(image_filename, 'rb') as f:
    encoded_image = base64.b64encode(f.read()).decode()

# -----------------------
# Build Layout
# -----------------------
app = JupyterDash('Project2Dashboard')

app.layout = html.Div([

    html.Center([
        html.Img(src=f'data:image/png;base64,{encoded_image}', style={'height':'100px'}),
        html.H1("Grazioso Salvare Dashboard by Maurice Sapp")
    ]),

    html.Hr(),

    # FILTER
    dcc.Dropdown(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster/Tracking', 'value': 'disaster'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        value='reset',
        clearable=False
    ),

    html.Hr(),

    # TABLE
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable="single",
        selected_rows=[0],
        page_size=10,
        sort_action="native",
        style_table={'overflowX': 'auto'}
    ),

    html.Br(),

    # GRAPH + MAP
    html.Div(style={'display': 'flex'}, children=[
        html.Div(id='graph-id', style={'width': '50%'}),
        html.Div(id='map-id', style={'width': '50%'})
    ])
])

# -----------------------
# FILTER TABLE
# -----------------------
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_table(filter_type):

    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Labrador|Retriever|Newfoundland", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd|Malamute|Husky|Sheepdog", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Bloodhound|Coonhound|Hound", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }

    else:
        return df.to_dict('records')

    filtered = shelter.read(query)
    dff = pd.DataFrame.from_records(filtered)

    if not dff.empty and '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')

# -----------------------
# CLEAN CHART
# -----------------------
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_chart(viewData):

    if viewData is None or len(viewData) == 0:
        return html.Div("No data to show chart")

    dff = pd.DataFrame.from_dict(viewData)

    dff['breed_clean'] = dff['breed'].apply(lambda x: x.split('/')[0].strip())

    breed_counts = dff['breed_clean'].value_counts().nlargest(10)

    fig = px.pie(
        names=breed_counts.index,
        values=breed_counts.values,
        title='Top 10 Breeds'
    )

    return dcc.Graph(figure=fig)

# -----------------------
# FIXED MAP (WORKS 100%)
# -----------------------
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return dl.Map(center=[30.75, -97.48], zoom=10, children=[dl.TileLayer()])

    dff = pd.DataFrame.from_dict(viewData)

    # Make sure lat/long exist
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return dl.Map(center=[30.75, -97.48], zoom=10, children=[dl.TileLayer()])

    # Drop bad rows
    dff = dff.dropna(subset=['location_lat', 'location_long'])

    if len(dff) == 0:
        return dl.Map(center=[30.75, -97.48], zoom=10, children=[dl.TileLayer()])

    # Safe row selection
    row = 0
    if index is not None and len(index) > 0 and index[0] < len(dff):
        row = index[0]

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']

    return dl.Map(
        style={'width': '100%', 'height': '500px'},
        center=[lat, lon],
        zoom=10,
        children=[
            dl.TileLayer(),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(dff.iloc[row]['breed']),
                dl.Popup([
                    html.H4("Animal Name"),
                    html.P(dff.iloc[row]['name'] if dff.iloc[row]['name'] else "No Name")
                ])
            ])
        ]
    )

# -----------------------
# RUN APP
# -----------------------
app.run_server(mode='inline')